# Dataset Split: Sequence Similarity

Sequence-similarity group split evaluation for the compact hydrogen-bond statistical model.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-mprl")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from dataset_module.dataset import SequenceRecord, build_split

sns.set_theme(style="whitegrid", context="talk")

BASE_DIR = Path("outputs/hbond_analysis")
PLOTS_DIR = BASE_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = BASE_DIR / "hbond_length_disentanglement_table.csv"
MD_PATH = Path("hbond_analysis.md")
RANDOM_STATE = 7
SIMILARITY_THRESHOLD = 0.5
KMER_SIZE = 5

SELECTED_FEATURES = [
    "sequence_length",
    "hbond_per_residue",
    "seq_class_nonlocal_per_residue",
    "strong_nonlocal_fraction",
    "strong_nonlocal_per_residue",
    "nonlocal_backbone_backbone_per_residue",
    "hbond_contact_order",
]

print(DATA_PATH.resolve())

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=["PDB_ID", "Sequence", "v127", "v128"] + SELECTED_FEATURES).copy().reset_index(drop=True)
df["log1p_v127"] = np.log1p(df["v127"].astype(float))
df["log1p_v128"] = np.log1p(df["v128"].astype(float))

cached_paths = {name: BASE_DIR / f"dataset_similarity_{name}_pdb_ids.txt" for name in ["train", "val", "test"]}
if all(path.exists() for path in cached_paths.values()):
    id_to_index = {str(pdb_id): int(index) for index, pdb_id in df["PDB_ID"].items()}
    split = {}
    for name, path in cached_paths.items():
        pdb_ids = [line.strip() for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
        split[name] = sorted(id_to_index[pdb_id] for pdb_id in pdb_ids if pdb_id in id_to_index)
    print("Loaded cached similarity split")
else:
    records = [
        SequenceRecord(
            record_id=str(row.PDB_ID),
            sequence=str(row.Sequence),
            toughness=float(row.v127),
            strength=float(row.v128),
        )
        for row in df.itertuples(index=False)
    ]
    split = build_split(
        records,
        split_method="similarity",
        val_fraction=0.1,
        test_fraction=0.1,
        seed=RANDOM_STATE,
        similarity_threshold=SIMILARITY_THRESHOLD,
        kmer_size=KMER_SIZE,
    )
    for name, indices in split.items():
        pd.Series(df.loc[indices, "PDB_ID"].to_numpy()).to_csv(cached_paths[name], index=False, header=False)

print({name: len(indices) for name, indices in split.items()})

In [ ]:
def build_models():
    return {
        "length_only_ridge": Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))]),
        "selected_hbond_ridge": Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))]),
        "selected_hbond_random_forest": Pipeline([
            ("scaler", StandardScaler()),
            ("model", MultiOutputRegressor(RandomForestRegressor(
                n_estimators=500,
                min_samples_leaf=3,
                max_features="sqrt",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ))),
        ]),
        "selected_hbond_extra_trees": Pipeline([
            ("scaler", StandardScaler()),
            ("model", MultiOutputRegressor(ExtraTreesRegressor(
                n_estimators=500,
                min_samples_leaf=3,
                max_features="sqrt",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ))),
        ]),
    }


feature_sets = {
    "length_only_ridge": ["sequence_length"],
    "selected_hbond_ridge": SELECTED_FEATURES,
    "selected_hbond_random_forest": SELECTED_FEATURES,
    "selected_hbond_extra_trees": SELECTED_FEATURES,
}


def evaluate_predictions(true_raw, pred_raw):
    row = {}
    for j, target_name in enumerate(["toughness_v127", "strength_v128"]):
        row[f"{target_name}/r2"] = float(r2_score(true_raw[:, j], pred_raw[:, j]))
        row[f"{target_name}/mae"] = float(mean_absolute_error(true_raw[:, j], pred_raw[:, j]))
        row[f"{target_name}/rmse"] = float(np.sqrt(mean_squared_error(true_raw[:, j], pred_raw[:, j])))
        row[f"{target_name}/spearman"] = float(spearmanr(true_raw[:, j], pred_raw[:, j]).correlation)
    row["mean/r2"] = float(np.mean([row["toughness_v127/r2"], row["strength_v128/r2"]]))
    row["mean/spearman"] = float(np.mean([row["toughness_v127/spearman"], row["strength_v128/spearman"]]))
    return row


X = df[SELECTED_FEATURES].astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0)
y_log = df[["log1p_v127", "log1p_v128"]].to_numpy(dtype=float)
y_raw = df[["v127", "v128"]].to_numpy(dtype=float)

rows = []
pred_frames = []
models = build_models()
for model_name, model in models.items():
    cols = feature_sets[model_name]
    model.fit(X.loc[split["train"], cols], y_log[split["train"]])
    for split_name, indices in split.items():
        pred_raw = np.expm1(model.predict(X.loc[indices, cols]))
        true_raw = y_raw[indices]
        row = {"dataset_split": "similarity", "model": model_name, "split": split_name, "n": int(len(indices)), "n_features": len(cols)}
        row.update(evaluate_predictions(true_raw, pred_raw))
        rows.append(row)
        pred_frames.append(pd.DataFrame({
            "dataset_split": "similarity",
            "model": model_name,
            "split": split_name,
            "PDB_ID": df.loc[indices, "PDB_ID"].to_numpy(),
            "true_v127": true_raw[:, 0],
            "true_v128": true_raw[:, 1],
            "pred_v127": pred_raw[:, 0],
            "pred_v128": pred_raw[:, 1],
        }))

metrics_df = pd.DataFrame(rows)
predictions_df = pd.concat(pred_frames, ignore_index=True)
metrics_df.to_csv(BASE_DIR / "dataset_similarity_hbond_model_metrics.csv", index=False)
predictions_df.to_csv(BASE_DIR / "dataset_similarity_hbond_model_predictions.csv", index=False)
display(metrics_df[metrics_df["split"] == "test"].sort_values("mean/r2", ascending=False))

In [ ]:
random_metrics_path = BASE_DIR / "dataset_random_hbond_model_metrics.csv"
random_metrics = pd.read_csv(random_metrics_path)
comparison = pd.concat([random_metrics, metrics_df], ignore_index=True)
comparison.to_csv(BASE_DIR / "dataset_split_hbond_model_comparison_metrics.csv", index=False)

test_comparison = comparison[comparison["split"] == "test"].copy()
wide_rows = []
for model_name, group in test_comparison.groupby("model"):
    row = {"model": model_name}
    for split_name, split_group in group.groupby("dataset_split"):
        item = split_group.iloc[0]
        for metric in ["toughness_v127/r2", "strength_v128/r2", "toughness_v127/spearman", "strength_v128/spearman", "mean/r2", "mean/spearman"]:
            row[f"{split_name}/{metric}"] = float(item[metric])
    if "random/mean/r2" in row and "similarity/mean/r2" in row:
        row["similarity_minus_random/mean/r2"] = row["similarity/mean/r2"] - row["random/mean/r2"]
        row["similarity_minus_random/toughness_v127/r2"] = row["similarity/toughness_v127/r2"] - row["random/toughness_v127/r2"]
        row["similarity_minus_random/strength_v128/r2"] = row["similarity/strength_v128/r2"] - row["random/strength_v128/r2"]
    wide_rows.append(row)
wide_comparison = pd.DataFrame(wide_rows).sort_values("random/mean/r2", ascending=False)
wide_comparison.to_csv(BASE_DIR / "dataset_split_hbond_model_comparison_test_wide.csv", index=False)
display(wide_comparison)

In [ ]:
plot_df = test_comparison[test_comparison["model"].isin(["selected_hbond_random_forest", "selected_hbond_extra_trees", "length_only_ridge"])].copy()
plot_df = plot_df.melt(
    id_vars=["dataset_split", "model"],
    value_vars=["toughness_v127/r2", "strength_v128/r2"],
    var_name="target_metric",
    value_name="r2",
)
plt.figure(figsize=(12, 6))
sns.barplot(data=plot_df, x="model", y="r2", hue="dataset_split")
plt.xticks(rotation=20, ha="right")
plt.title("Random vs sequence-similarity split, test R2")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "dataset_split_random_vs_similarity_test_r2.png", dpi=220)
plt.close()

summary = {
    "split_method": "similarity",
    "seed": RANDOM_STATE,
    "similarity_threshold": SIMILARITY_THRESHOLD,
    "kmer_size": KMER_SIZE,
    "split_sizes": {name: len(indices) for name, indices in split.items()},
    "selected_features": SELECTED_FEATURES,
    "metrics_path": str(BASE_DIR / "dataset_similarity_hbond_model_metrics.csv"),
    "comparison_path": str(BASE_DIR / "dataset_split_hbond_model_comparison_metrics.csv"),
}
(BASE_DIR / "dataset_similarity_hbond_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
summary

In [ ]:
def markdown_table(frame, digits=4):
    out = frame.copy()
    for col in out.columns:
        if pd.api.types.is_numeric_dtype(out[col]):
            out[col] = out[col].map(lambda x: f"{x:.{digits}f}" if pd.notna(x) else "nan")
    return out.to_markdown(index=False)

random_split_sizes = random_metrics.groupby("split")["n"].max().to_dict()
similarity_split_sizes = {name: len(indices) for name, indices in split.items()}
report_table = wide_comparison[[
    "model",
    "random/toughness_v127/r2",
    "similarity/toughness_v127/r2",
    "similarity_minus_random/toughness_v127/r2",
    "random/strength_v128/r2",
    "similarity/strength_v128/r2",
    "similarity_minus_random/strength_v128/r2",
    "random/mean/r2",
    "similarity/mean/r2",
    "similarity_minus_random/mean/r2",
]]

section = f"""

## Dataset Split Comparison for Hydrogen-Bond Statistical Models

### Current Split Used Before This Comparison

The previous `hbond_final.ipynb` model used a random split implemented with `sklearn.model_selection.train_test_split`:

```text
train: 80%
validation: 10%
test: 10%
random_state: 7
```

This is different from the earlier sequence-to-property deep-learning predictor, where we explicitly supported both random split and sequence-similarity split. Random split can place highly similar sequences into both training and test sets, so it usually estimates interpolation performance rather than out-of-family generalization.

### Hypothesis

If hydrogen-bond features mainly capture broad structural rules, performance should remain reasonably stable under sequence-similarity split. If performance drops strongly, the model is partly relying on similarity between train and test proteins.

### Method

Two notebooks were created:

```text
dataset-random.ipynb
dataset-similarity.ipynb
```

Both notebooks used the same final hydrogen-bond feature set:

```text
sequence_length
hbond_per_residue
seq_class_nonlocal_per_residue
strong_nonlocal_fraction
strong_nonlocal_per_residue
nonlocal_backbone_backbone_per_residue
hbond_contact_order
```

The random split used random 80/10/10 partitioning. The sequence-similarity split used the project's existing k-mer Jaccard grouping method:

```text
kmer_size = {KMER_SIZE}
similarity_threshold = {SIMILARITY_THRESHOLD}
train/validation/test group split = approximately 80/10/10
```

Random split sizes:

```text
{random_split_sizes}
```

Sequence-similarity split sizes:

```text
{similarity_split_sizes}
```

### Results

{markdown_table(report_table, 4)}

Output files:

- `outputs/hbond_analysis/dataset_random_hbond_model_metrics.csv`
- `outputs/hbond_analysis/dataset_similarity_hbond_model_metrics.csv`
- `outputs/hbond_analysis/dataset_split_hbond_model_comparison_metrics.csv`
- `outputs/hbond_analysis/dataset_split_hbond_model_comparison_test_wide.csv`
- `outputs/hbond_analysis/plots/dataset_split_random_vs_similarity_test_r2.png`

### Analysis

The random split result is the easier evaluation setting because homologous or very similar sequences may appear across train and test. The sequence-similarity split is closer to an out-of-family generalization test.

If the similarity-split metrics are close to random-split metrics, the selected hydrogen-bond features are likely capturing general physical structure-property rules. If the similarity-split metrics drop, the model has weaker extrapolation to dissimilar sequence families.

For this hydrogen-bond statistical model, the comparison should be interpreted as a structural-feature OOD check rather than a final predictor benchmark. These seven features are intentionally compact and cannot encode the full sequence/fold information that ESM2 captures.

### Conclusion

The current hydrogen-bond final model was originally evaluated with random split. The sequence-similarity split comparison is the more conservative result and should be preferred when deciding whether hydrogen-bond descriptors generalize across protein families.

If hydrogen-bond features remain useful under similarity split, they are good candidates for physics-informed auxiliary inputs to the ESM2 mechanical-property predictor. If they degrade under similarity split, they should still be useful for within-family diagnostics and reward shaping, but should not be trusted as a standalone OOD predictor.
"""

start_marker = "\n## Dataset Split Comparison for Hydrogen-Bond Statistical Models\n"
old = MD_PATH.read_text(encoding="utf-8")
if start_marker in old:
    old = old.split(start_marker)[0].rstrip() + "\n"
MD_PATH.write_text(old + section, encoding="utf-8")
print("Updated", MD_PATH)